In [0]:
%run ../setup_nautiq_dev

# Nautiq - setup del entorno

Configuracion centralizada para:

- Acceso seguro a Bronze mediante SAS Token.
- Rutas de los topics AIS.
- Estado tecnico de Auto Loader.
- Tablas Silver y Gold.
- Cambio futuro entre tablas administradas y ADLS externo.

DataFrame[]

NAUTIQ - CONFIGURACION DEL ENTORNO
Environment: dev
Catalog: masterxyz002dbr
Target storage mode: managed
Ops volume: /Volumes/masterxyz002dbr/ops/nautiq_dev

[OK] ais_positions: 15 elementos encontrados
[OK] ais_static: 23 elementos encontrados

Silver tables:
  - masterxyz002dbr.silver.ais_positions_dev
  - masterxyz002dbr.silver.ais_static_dev

Gold tables:
  - masterxyz002dbr.gold.vessel_operation_times

Data quality table:
  - masterxyz002dbr.ops.data_quality_checks

[OK] Setup completado correctamente.


In [0]:
# Crea la tabla Delta Silver con informacion del buque y del viaje.

location_clause = (
    "" if static_target_path is None else f"LOCATION '{static_target_path}'"
)

spark.sql(
    f"""
CREATE TABLE IF NOT EXISTS {static_target_table} (
    mmsi BIGINT COMMENT 'Maritime Mobile Service Identity; partition key en Kafka de origen',
    imo BIGINT COMMENT 'Número IMO del buque',
    vessel_name STRING COMMENT 'Nombre del buque',
    call_sign STRING COMMENT 'Indicativo de llamada del buque',
    ship_type_code INT COMMENT 'Código AIS del tipo de buque',
    vessel_length_meters INT COMMENT 'Eslora del buque en metros',
    vessel_beam_meters INT COMMENT 'Manga del buque en metros',
    vessel_draught_meters DOUBLE COMMENT 'Calado del buque en metros',
    destination_raw STRING COMMENT 'Destino declarado en el mensaje AIS',
    eta_raw STRING COMMENT 'ETA declarada en el mensaje AIS',
    destination_port_code STRING COMMENT 'Código UN/LOCODE normalizado para uno de los puertos objetivo',
    destination_port_name STRING COMMENT 'Nombre normalizado del puerto objetivo identificado desde destination_raw',
    correlation_id STRING COMMENT 'Identificador de correlacion y trazabilidad del evento',
    bronze_ingested_timestamp TIMESTAMP COMMENT 'Instante de ingestión informado en _ingested_at del fichero Bronze',
    kafka_ingestion_timestamp TIMESTAMP COMMENT 'Instante de ingestión informado en Bronze por el pipeline Kafka/Flink',
    kafka_partition BIGINT COMMENT 'Partición Kafka de origen',
    kafka_offset BIGINT COMMENT 'Offset Kafka de origen dentro de la partición',
    flink_processing_timestamp TIMESTAMP COMMENT 'Instante de procesamiento informado por Flink',
    schema_version BIGINT COMMENT 'Versión del esquema informada en Bronze',
    bronze_partition_date DATE COMMENT 'Fecha extraída de la partición _dt del fichero Bronze',
    source_file STRING COMMENT 'Ruta del fichero Parquet de origen en Bronze',
    silver_processed_timestamp TIMESTAMP COMMENT 'Instante en que Databricks procesó el registro en Silver'
)
USING DELTA
COMMENT 'Historico normalizado de informacion estatica y viaje AIS'
{location_clause}
"""
)

DataFrame[]

In [0]:
# Lee solamente los nuevos Parquet compactados del topic static.

static_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", static_schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("pathGlobFilter", "compacted-part-*")
    .load(static_source)
    .selectExpr("*", "_metadata.file_path AS source_file")
)

static_raw.createOrReplaceTempView("vw_bronze_static_stream")

In [0]:
# Normaliza columnas, puerto objetivo y ETA, y elimina duplicados Kafka.

static_silver = (
    spark.sql(fr"""
        WITH base AS (
            SELECT
                CAST(mmsi AS BIGINT) AS mmsi,
                CAST(imo AS BIGINT) AS imo,
                NULLIF(TRIM(name), '') AS vessel_name,
                NULLIF(TRIM(callsign), '') AS call_sign,
                CAST(ship_type AS INT) AS ship_type_code,
                CAST(length_m AS INT) AS vessel_length_meters,
                CAST(beam_m AS INT) AS vessel_beam_meters,
                CAST(draught_m AS DOUBLE) AS vessel_draught_meters,
                NULLIF(TRIM(destination), '') AS destination_raw,
                REGEXP_REPLACE(UPPER(TRIM(destination)), '[^A-Z0-9]', '') AS destination_clean,
                eta AS eta_raw,
                CAST(correlation_id AS STRING) AS correlation_id,
                TRY_TO_TIMESTAMP(_ingested_at) AS bronze_ingested_timestamp,
                CAST(_kafka_ingestion_time AS TIMESTAMP) AS kafka_ingestion_timestamp,
                CAST(_kafka_partition AS BIGINT) AS kafka_partition,
                CAST(_kafka_offset AS BIGINT) AS kafka_offset,
                CAST(_flink_processing_time AS TIMESTAMP) AS flink_processing_timestamp,
                CAST(_schema_version AS BIGINT) AS schema_version,
                CAST(REGEXP_EXTRACT(source_file, '_dt=([0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}})', 1) AS DATE) AS bronze_partition_date,
                source_file
            FROM vw_bronze_static_stream
            WHERE mmsi IS NOT NULL
              AND correlation_id IS NOT NULL
              AND _kafka_partition IS NOT NULL
              AND _kafka_offset IS NOT NULL
        ),
        normalized AS (
            SELECT *
            FROM base
            WHERE bronze_partition_date >= DATE '{bronze_start_date}'
        ),
        ports AS (
            SELECT *,
                CASE
                    WHEN destination_clean RLIKE '^(ESVLC|VALENCIA|VAL)$' THEN 'ESVLC'
                    WHEN destination_clean RLIKE '^(ESBCN|BARCELONA|BCN)$' THEN 'ESBCN'
                    WHEN destination_clean RLIKE '^(ESALG|ALGECIRAS|ALG)$' THEN 'ESALG'
                END AS destination_port_code
            FROM normalized
        )
        SELECT
            mmsi, imo, vessel_name, call_sign, ship_type_code,
            vessel_length_meters, vessel_beam_meters, vessel_draught_meters,
            destination_raw, eta_raw,
            destination_port_code,
            CASE destination_port_code
                WHEN 'ESVLC' THEN 'Valencia'
                WHEN 'ESBCN' THEN 'Barcelona'
                WHEN 'ESALG' THEN 'Algeciras'
            END AS destination_port_name,
            correlation_id,
            bronze_ingested_timestamp, kafka_ingestion_timestamp,
            kafka_partition, kafka_offset, flink_processing_timestamp,
            schema_version, bronze_partition_date, source_file,
            CURRENT_TIMESTAMP() AS silver_processed_timestamp
        FROM ports
    """)
    .dropDuplicates(["kafka_partition", "kafka_offset"])
)

In [0]:
# Escribe los registros validos en Silver.

query = (
    static_silver.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", static_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(static_target_table)
)

query.awaitTermination()

print(f"[OK] Tabla actualizada: {static_target_table}")

[OK] Tabla actualizada: masterxyz002dbr.silver.ais_static
